In [1]:
import torch
import numpy
import matplotlib
from GPTModel import *
from GPTDataset import *
from utility import *
GPT_CONFIG = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers":12,
    "drop_rate":0.1,
    "qkv_bias":False
}

In [2]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG)
model.eval()

GPTModel(
  (embeddings): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): NormalizationLayer()
      (norm2): NormalizationLayer()
      (drop_shrtct): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): 

In [3]:
text_to_token("Hello World!",tiktoken.get_encoding("gpt2"))

tensor([[15496,  2159,     0]])

In [4]:
tokenizer = tiktoken.get_encoding("gpt2")
tokens = text_to_token("Every effort moves you",tokenizer)
tokens


tensor([[6109, 3626, 6100,  345]])

In [5]:
new_text = generate_text_simple(model,tokens,10,4)


In [6]:
token_to_text(new_text,tokenizer)


'Every effort moves you renting Hue Pleasant Wisdom Wisdom�� businessmenOTT�heart'

In [7]:
raw_text=open("the-verdict.txt").read()
raw_text[:99]

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no '

In [8]:
train_ratio = 0.9
split_idx = int(train_ratio*len(raw_text))
train_data=raw_text[:split_idx]
val_data= raw_text[split_idx:]

In [10]:
train_loader = create_dataloader(train_data,
                                 batch_size=2,
                                 window=GPT_CONFIG["context_length"],
                                 stride=GPT_CONFIG["context_length"],
                                 drop_last=True,
                                 shuffle=True)
val_loader = create_dataloader(val_data,
                                 batch_size=2,
                                 window=GPT_CONFIG["context_length"],
                                 stride=GPT_CONFIG["context_length"],
                                 drop_last=False,
                                 shuffle=False)

In [11]:
print("Train Loader")
for x,y in train_loader:
    pass
print(x.shape,y.shape)

Train Loader
torch.Size([2, 256]) torch.Size([2, 256])


In [40]:
device = torch.device("gpu")

RuntimeError: Expected one of cpu, cuda, ipu, xpu, mkldnn, opengl, opencl, ideep, hip, ve, fpga, maia, xla, lazy, vulkan, mps, meta, hpu, mtia, privateuseone device type at start of device string: gpu

In [41]:
def calc_loss_batch(input_batch,target_batch,model,device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1),
        target_batch.flatten()
    )    
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batces = min(num_batches,len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i<num_batches:
            loss = calc_loss_batch(input_batch,target_batch,model,device)
            total_loss += loss
        else:
            break
    return total_loss/num_batches


In [42]:
def train_model(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0,-1
    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch,target_batch,model,device)
            loss.backward()
            optimizer.step()
            tokens_seen+= input_batch.numel()
            global_step+=1

            if global_step%eval_freq==0:
                train_loss,val_loss = evaluate_model(model,train_loader,val_loader,device,eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"EP {epoch+1} (step {global_step:06d}): "
                      f" Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")
            generate_and_print_sample(model,tokenizer,device,start_context)
    return train_losses, val_losses, track_tokens_seen
    

In [43]:
def evaluate_model(model,train_loader, val_loader, device, itr):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader,model,device,itr)
        val_loss = calc_loss_loader(val_loader,model,device,itr)
    model.train()
    return train_loss,val_loss

def generate_and_print_sample(model,tokenizer,device,start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token(start_context,tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(model=model,idx=encoded,max_new_tokens=50,context_size=context_size)
    decoded_text=token_to_text(token_ids,tokenizer)
    print(decoded_text.replace("\n"," "))
    model.train()

In [44]:
model = GPTModel(GPT_CONFIG)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(),lr=0.0004,weight_decay=0.1)
num_epochs=10


In [45]:
train_losses, val_losses, tokens_seen = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    eval_freq=5,
    eval_iter=5,
    start_context="Every effort moves you",
    tokenizer=tokenizer
)


EP 1 (step 000000):  Train loss 9.750, Val loss 2.005
Every effort moves you                                                  
Every effort moves you,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Every effort moves you,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Every effort moves you,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Every effort moves you,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
EP 1 (step 000005):  Train loss 8.113, Val loss 1.681
Every effort moves you,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Every effort moves you,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Every effort moves you the the, the the, the, the, the, the the, the, the the, the, the, the, the the, the the, the the the, the, the, the the the, the, the the, the
Every effort moves you                                                  
Every effort moves you.                                                 
EP 2 (step 000010):  Train loss 6.796, Val loss 1.429
Every effort mo